In [2]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q transformers accelerate scikit-learn

In [4]:
# ── Cell 2: Mount & Load Data ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/gdrive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/gdrive/Shareddrives/FML_FINAL/Data/movies_with_roi.csv')
df['log_roi'] = pd.to_numeric(df['log_roi'], errors='coerce')
df = df[['overview', 'log_roi']].dropna()
df = df[df['overview'].str.strip() != ''].reset_index(drop=True)

print(f"Total usable rows : {len(df)}")
print(f"log_roi stats:\n{df['log_roi'].describe()}")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Total usable rows : 4278
log_roi stats:
count    4278.000000
mean        0.380629
std         1.791207
min       -13.661360
25%        -0.302626
50%         0.641102
75%         1.360531
max        14.914124
Name: log_roi, dtype: float64


In [8]:
# ── Cell 3: Split ─────────────────────────────────────────────────────────────
df['roi_quartile'] = pd.qcut(df['log_roi'], q=4, labels=False)

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['roi_quartile']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['roi_quartile']
)

train_df = train_df.drop(columns='roi_quartile').reset_index(drop=True)
val_df   = val_df.drop(columns='roi_quartile').reset_index(drop=True)
test_df  = test_df.drop(columns='roi_quartile').reset_index(drop=True)

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

Train : 3422
Val   : 428
Test  : 428


In [9]:
# ── Cell 4: Device + Tokenizer ────────────────────────────────────────────────
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Check token lengths so we set MAX_LENGTH correctly
lengths = [len(tokenizer.encode(t)) for t in df['overview']]
print(f"\nToken lengths:")
print(f"  Mean   : {np.mean(lengths):.0f}")
print(f"  Max    : {np.max(lengths)}")
print(f"  99th % : {np.percentile(lengths, 99):.0f}")

Device : cuda
GPU    : Tesla T4

Token lengths:
  Mean   : 60
  Max    : 234
  99th % : 158


In [10]:
# ── Cell 5: Dataset ───────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 256

class OverviewDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.encodings = tokenizer(
            list(df['overview']),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        self.labels = torch.tensor(df['log_roi'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

train_dataset = OverviewDataset(train_df, tokenizer)
val_dataset   = OverviewDataset(val_df,   tokenizer)
test_dataset  = OverviewDataset(test_df,  tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 107
Val batches   : 7
Test batches  : 7


In [11]:
# ── Cell 6: Model ─────────────────────────────────────────────────────────────
class RobertaForROI(nn.Module):
    def __init__(self, dropout=0.2):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        hidden_size  = self.roberta.config.hidden_size  # 768

        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs   = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.pooler_output          # [CLS] token → (batch, 768)
        pred      = self.regression_head(cls_embed).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.MSELoss()(pred, labels)

        return {'loss': loss, 'logits': pred}


model = RobertaForROI(dropout=0.2).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Model device     : {next(model.parameters()).device}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params     : 124,859,009
Trainable params : 124,859,009
Model device     : cuda:0


In [12]:
# ── Cell 7: Training ──────────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 10
LR     = 2e-5

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

MODEL_SAVE_PATH = '/content/gdrive/Shareddrives/FML_FINAL/Models/roberta_roi_best.pt'

import os
os.makedirs('/content/gdrive/Shareddrives/FML_FINAL/Models', exist_ok=True)


def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            out            = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss    += out['loss'].item()
            all_preds.extend(out['logits'].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mae        = np.mean(np.abs(all_preds - all_labels))
    ss_res     = np.sum((all_labels - all_preds) ** 2)
    ss_tot     = np.sum((all_labels - np.mean(all_labels)) ** 2)
    r2         = 1 - ss_res / ss_tot
    return total_loss / len(loader), mae, r2


best_val_r2 = -999

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out['loss']
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if step % 20 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # ── Validate ──────────────────────────────────────────────────────────────
    train_loss            = total_train_loss / len(train_loader)
    val_loss, val_mae, val_r2 = evaluate(val_loader)

    print(f"""
{'='*50}
Epoch {epoch+1}/{EPOCHS}
  Train Loss : {train_loss:.4f}
  Val Loss   : {val_loss:.4f}
  Val MAE    : {val_mae:.4f}
  Val R²     : {val_r2:.4f}
{'='*50}
""")

    # ── Save best ─────────────────────────────────────────────────────────────
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  💾 New best saved — Val R²: {val_r2:.4f}")

  Epoch 1 | Step 0/107 | Loss: 1.2494
  Epoch 1 | Step 20/107 | Loss: 2.4841
  Epoch 1 | Step 40/107 | Loss: 3.9169
  Epoch 1 | Step 60/107 | Loss: 4.0665
  Epoch 1 | Step 80/107 | Loss: 3.8419
  Epoch 1 | Step 100/107 | Loss: 1.1108

Epoch 1/10
  Train Loss : 3.2170
  Val Loss   : 3.3077
  Val MAE    : 1.2316
  Val R²     : 0.0033

  💾 New best saved — Val R²: 0.0033
  Epoch 2 | Step 0/107 | Loss: 2.1160
  Epoch 2 | Step 20/107 | Loss: 5.5101
  Epoch 2 | Step 40/107 | Loss: 0.9191
  Epoch 2 | Step 60/107 | Loss: 5.6760
  Epoch 2 | Step 80/107 | Loss: 4.9744
  Epoch 2 | Step 100/107 | Loss: 1.3969

Epoch 2/10
  Train Loss : 3.1135
  Val Loss   : 3.2356
  Val MAE    : 1.1945
  Val R²     : 0.0235

  💾 New best saved — Val R²: 0.0235
  Epoch 3 | Step 0/107 | Loss: 1.3079
  Epoch 3 | Step 20/107 | Loss: 1.3839
  Epoch 3 | Step 40/107 | Loss: 1.5567
  Epoch 3 | Step 60/107 | Loss: 3.6799
  Epoch 3 | Step 80/107 | Loss: 1.9356
  Epoch 3 | Step 100/107 | Loss: 1.4626

Epoch 3/10
  Train Loss

In [13]:
# ── Cell 8: Test Evaluation ───────────────────────────────────────────────────
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
test_loss, test_mae, test_r2 = evaluate(test_loader)

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Final Test Set Results
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Test Loss : {test_loss:.4f}
  Test MAE  : {test_mae:.4f}
  Test R²   : {test_r2:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Best Val R² : {best_val_r2:.4f}
  Model saved : {MODEL_SAVE_PATH}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Final Test Set Results
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Test Loss : 3.3644
  Test MAE  : 1.1795
  Test R²   : 0.0337
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Best Val R² : 0.0235
  Model saved : /content/gdrive/Shareddrives/FML_FINAL/Models/roberta_roi_best.pt
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [14]:
# ── Cell 10: Predict on new overviews ─────────────────────────────────────────
def predict_roi(overviews):
    model.eval()
    encodings = tokenizer(
        overviews,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    with torch.no_grad():
        out = model(
            input_ids=encodings['input_ids'].to(device),
            attention_mask=encodings['attention_mask'].to(device),
        )
    return out['logits'].cpu().numpy()


test_overviews = [
    "A young farm boy discovers he is destined to save the galaxy from an evil empire.",
    "Two strangers fall in love on a doomed ocean liner crossing the Atlantic.",
]
preds = predict_roi(test_overviews)
for overview, pred in zip(test_overviews, preds):
    print(f"Overview  : {overview[:70]}...")
    print(f"log_roi   : {pred:.4f}  →  ROI: {np.expm1(pred)*100:.1f}%\n")

Overview  : A young farm boy discovers he is destined to save the galaxy from an e...
log_roi   : 0.3410  →  ROI: 40.6%

Overview  : Two strangers fall in love on a doomed ocean liner crossing the Atlant...
log_roi   : 0.1720  →  ROI: 18.8%



In [15]:
# ── Pick 2 real examples from test set and compare predicted vs actual ─────────
import numpy as np

# Grab 2 samples from test set — one high ROI, one low ROI
high_roi_idx = test_df['log_roi'].idxmax()
low_roi_idx  = test_df['log_roi'].idxmin()

samples = test_df.loc[[high_roi_idx, low_roi_idx]].reset_index(drop=True)

# Predict
model.eval()
encodings = tokenizer(
    list(samples['overview']),
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors='pt',
)
with torch.no_grad():
    out = model(
        input_ids=encodings['input_ids'].to(device),
        attention_mask=encodings['attention_mask'].to(device),
    )
preds = out['logits'].cpu().numpy()

# Print results
for i, row in samples.iterrows():
    actual_log_roi  = row['log_roi']
    pred_log_roi    = preds[i]
    actual_roi_pct  = np.expm1(actual_log_roi) * 100
    pred_roi_pct    = np.expm1(pred_log_roi)   * 100

    print(f"{'='*60}")
    print(f"Overview  : {row['overview']}")
    print(f"{'─'*60}")
    print(f"Actual log_roi    : {actual_log_roi:.4f}  →  ROI: {actual_roi_pct:.1f}%")
    print(f"Predicted log_roi : {pred_log_roi:.4f}  →  ROI: {pred_roi_pct:.1f}%")
    print(f"Error             : {abs(actual_log_roi - pred_log_roi):.4f} log_roi units")
    print()

# Also grab a random middle one
mid_idx = test_df.index[len(test_df)//2]
mid     = test_df.loc[[mid_idx]].reset_index(drop=True)
enc     = tokenizer(list(mid['overview']), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
with torch.no_grad():
    mid_pred = model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device))['logits'].cpu().numpy()

print(f"{'='*60}")
print(f"[MIDDLE ROI EXAMPLE]")
print(f"Overview  : {mid['overview'].values[0]}")
print(f"{'─'*60}")
print(f"Actual log_roi    : {mid['log_roi'].values[0]:.4f}  →  ROI: {np.expm1(mid['log_roi'].values[0])*100:.1f}%")
print(f"Predicted log_roi : {mid_pred[0]:.4f}  →  ROI: {np.expm1(mid_pred[0])*100:.1f}%")
print(f"Error             : {abs(mid['log_roi'].values[0] - mid_pred[0]):.4f} log_roi units")

Overview  : KDTV puts together a holiday party, but one accident derails it all.
────────────────────────────────────────────────────────────
Actual log_roi    : 11.9184  →  ROI: 14999900.0%
Predicted log_roi : 0.7591  →  ROI: 113.6%
Error             : 11.1593 log_roi units

Overview  : Harassed by bullies because of his mild autism, teen Ben finds refuge in an online computer game, which leads him to his virtual dream girl, Scarlite. Together, the odd couple seeks revenge against Ben's tormentors.
────────────────────────────────────────────────────────────
Actual log_roi    : -10.9251  →  ROI: -100.0%
Predicted log_roi : 0.7597  →  ROI: 113.8%
Error             : 11.6849 log_roi units

[MIDDLE ROI EXAMPLE]
Overview  : G.G. Sparrow faces off with her choir's newly appointed director, Vi Rose Hill, over the group's direction as they head into a national competition.
────────────────────────────────────────────────────────────
Actual log_roi    : 0.2202  →  ROI: 24.6%
Predicted log_roi